<a href="https://colab.research.google.com/github/sebastiangome/latam-agente/blob/main/LATAM_RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Agente de Cuenta LATAM Airlines — Pipeline RAG (ISY0101, EP1)

Notebook del encargo **Diseño de Solución con LLM y RAG**, construido con el mismo stack visto en
clases (RA1/IL1.3 y RA1/IL1.1 del curso):

- **LangChain** como framework de orquestación.
- **Groq** (`ChatOpenAI` apuntando a su endpoint compatible con OpenAI) para el LLM de chat.
- **Gemini** (`GoogleGenerativeAIEmbeddings`) para los embeddings, ya que Groq no expone ese endpoint.
- **FAISS** (vía `langchain_community.vectorstores`) como base de datos vectorial.

Caso: LATAM Airlines — chatbot que responde consultas de pasajeros sobre cambios de vuelo,
equipaje y Millas LATAM Pass, combinando:

- **Fuente interna (simulada)**: perfil de reserva del pasajero (tarifa, cabina, ruta, categoría
  LATAM Pass, millas).
- **Fuente externa (real)**: resúmenes propios de páginas oficiales del Centro de Ayuda de LATAM
  (no se reproduce el texto original; se cita la fuente en cada chunk y en el informe).

> **Credenciales necesarias** (igual que en los notebooks de clase):
> `LLM_API_KEY` (Groq, gratis en [console.groq.com/keys](https://console.groq.com/keys)) y
> `GOOGLE_API_KEY` (Gemini, gratis en [aistudio.google.com/apikey](https://aistudio.google.com/apikey)).
> Cárgalas como **Secrets de Colab** (icono ) con esos nombres exactos, con el interruptor
> "Notebook access" activado. Nunca las escribas directo en una celda.


## MÓDULO 1 — Setup y Configuración de Modelos

**Alcance técnico:**
- Configuración de dependencias (Google Colab / Entorno local).
- Carga segura de credenciales (`LLM_API_KEY`, `GOOGLE_API_KEY`) desde Secrets o `.env`.
- Inicialización del LLM (`ChatOpenAI` con endpoint Groq) y Embeddings (`GoogleGenerativeAIEmbeddings`).

## 1. Instalación de dependencias

In [52]:
import sys
if "google.colab" in sys.modules:
    !pip install -q faiss-cpu langchain langchain-community langchain-core langchain-google-genai langchain-openai openai python-dotenv


## 2. Credenciales

Mismo patrón que los notebooks del curso: intenta leer los Secrets de Colab; si no está en
Colab, cae a un archivo `.env` local.

In [53]:
import os
try:
    from google.colab import userdata
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LLM_BASE_URL", "LLM_MODEL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "llama-3.3-70b-versatile")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

import random
import re
import numpy as np
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

try:
    from langchain_core.messages import HumanMessage, SystemMessage
except ImportError:
    from langchain.schema import HumanMessage, SystemMessage

random.seed(42)
np.random.seed(42)


## 3. Configuración de los modelos (chat + embeddings)

In [54]:
llm = ChatOpenAI(
    base_url=os.getenv("LLM_BASE_URL", "https://api.groq.com/openai/v1"),
    api_key=os.getenv("LLM_API_KEY"),
    model=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
    temperature=0.2,
    max_tokens=500,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

print("[OK] LLM (Groq LLaMA-3.3-70B) y embeddings (Gemini text-embedding-004) configurados.")


[OK] LLM (Groq LLaMA-3.3-70B) y embeddings (Gemini text-embedding-004) configurados.


## MÓDULO 2 — Datos Internos Simulados y Políticas Oficiales Externas

**Alcance técnico:**
- Modelado y generación del dataset sintético de reservas de pasajeros (`reservas_latam.csv`).
- Construcción de perfiles narrativos individuales aptos para búsqueda RAG.
- Curaduría de chunks oficiales de políticas públicas del Centro de Ayuda de LATAM con metadata y URLs.

## 4. Generación de datos sintéticos de reservas

Se simulan perfiles de reserva de pasajeros con los campos relevantes para responder consultas
de cambios, equipaje y millas: código de reserva, tarifa, cabina, ruta y categoría LATAM Pass.

In [55]:
import os
TARIFAS = ["Light", "Plus", "Top", "Full"]
CABINAS = ["Economy", "Premium Economy", "Premium Business"]
CATEGORIAS_PASS = ["Red", "Silver", "Gold", "Platinum", "Black", "Black Signature"]
RUTAS = ["Santiago-Lima", "Santiago-Bogota", "Santiago-Sao Paulo", "Santiago-Miami", "Santiago-Buenos Aires"]

def generar_codigo_reserva():
    letras = "".join(random.choices("ABCDEFGHJKLMNPQRSTUVWXYZ", k=3))
    numeros = "".join(random.choices("0123456789", k=3))
    return letras + numeros

def generar_reservas(n=200):
    filas = []
    codigos_usados = set()
    for _ in range(n):
        codigo = generar_codigo_reserva()
        while codigo in codigos_usados:
            codigo = generar_codigo_reserva()
        codigos_usados.add(codigo)

        tarifa = random.choice(TARIFAS)
        cabina = random.choice(CABINAS)
        ruta = random.choice(RUTAS)
        categoria = np.random.choice(CATEGORIAS_PASS, p=[0.45, 0.25, 0.15, 0.08, 0.05, 0.02])
        millas = int(np.random.exponential(15000))

        filas.append({
            "codigo_reserva": codigo,
            "tarifa": tarifa,
            "cabina": cabina,
            "ruta": ruta,
            "categoria_pass": categoria,
            "millas_acumuladas": millas,
        })
    return pd.DataFrame(filas)

df_reservas = generar_reservas(200)
os.makedirs("data", exist_ok=True)
df_reservas.to_csv("data/reservas_latam.csv", index=False)
df_reservas.to_csv("reservas_latam.csv", index=False)
print(f"[OK] {len(df_reservas)} reservas generadas y guardadas en data/reservas_latam.csv")
df_reservas.head()


[OK] 200 reservas generadas y guardadas en data/reservas_latam.csv


,codigo_reserva,tarifa,cabina,ruta,categoria_pass,millas_acumuladas
0,RAG276,Light,Premium Business,Santiago-Miami,Red,45151
1,ACF657,Full,Economy,Santiago-Miami,Gold,13694
2,QVA863,Plus,Economy,Santiago-Sao Paulo,Red,2543
3,CKJ320,Full,Premium Business,Santiago-Lima,Red,30168
4,ZKP868,Plus,Premium Business,Santiago-Lima,Silver,18468


## 5. Construcción de perfiles narrativos de reserva (chunks internos)

In [56]:
def construir_perfil_reserva(row):
    return (
        f"Reserva {row['codigo_reserva']}: tarifa {row['tarifa']}, cabina {row['cabina']}, "
        f"ruta {row['ruta']}, categoria LATAM Pass {row['categoria_pass']}, "
        f"{row['millas_acumuladas']} millas acumuladas."
    )

df_reservas["perfil_narrativo"] = df_reservas.apply(construir_perfil_reserva, axis=1)
chunks_reservas = df_reservas["perfil_narrativo"].tolist()
print(chunks_reservas[0])


Reserva RAG276: tarifa Light, cabina Premium Business, ruta Santiago-Miami, categoria LATAM Pass Red, 45151 millas acumuladas.


## 6. Fuente externa: resúmenes propios de políticas oficiales de LATAM (chunks externos)

Cada chunk resume, con palabras propias, el contenido de una página oficial del Centro de Ayuda
de LATAM. Se guarda la URL de origen aparte, para citarla en el informe (referencias APA) y no
se mezcla con el texto del chunk que va al vector store.

In [57]:
# >>> Compañero B <<<
documentos_externos = [
    {
        "id": "ext_cambios_01",
        "tema": "cambios",
        "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/cambios/pasajes/cambiar-vuelo-fecha-pasaje",
        "texto": (
            "Un pasajero puede cambiar la fecha o el vuelo de su pasaje siempre que las condiciones "
            "de su tarifa lo permitan. El cambio debe solicitarse antes de la salida del vuelo original; "
            "en algunos casos tambien es posible cambiarlo despues de iniciado el viaje si la tarifa lo "
            "autoriza. El proceso se realiza desde la seccion 'Mis Viajes', ingresando el codigo de "
            "reserva y el apellido del pasajero."
        ),
    },
    {
        "id": "ext_cambios_02",
        "tema": "cambios",
        "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/problemas-vuelo/cambio-itinerario/vuelo-adelantado",
        "texto": (
            "Si LATAM adelanta un vuelo 16 minutos o mas y el pasajero no esta conforme con el nuevo "
            "horario, puede cambiar la hora o fecha del vuelo sin costo, o solicitar la devolucion del "
            "pasaje, siempre que mantenga el mismo destino y cabina del vuelo original. El plazo para "
            "solicitar este cambio es de hasta 12 meses desde la fecha del vuelo original comprado."
        ),
    },
    {
        "id": "ext_equipaje_01",
        "tema": "equipaje",
        "fuente_url": "https://www.latamairlines.com/us/es/centro-ayuda/preguntas/equipaje",
        "texto": (
            "La franquicia de equipaje facturado depende de la tarifa, cabina y ruta del pasaje. En "
            "general, la cabina Economy permite una cantidad de piezas menor que Premium Economy o "
            "Premium Business. Ademas del equipaje facturado, se permite equipaje de mano y un articulo "
            "personal, con limites de peso y tamano que tambien varian segun la tarifa contratada. Es "
            "importante verificar la franquicia especifica en el sitio oficial antes de viajar, ya que "
            "estas politicas pueden cambiar (por ejemplo, LATAM ha reducido en el pasado la cantidad de "
            "piezas gratuitas en algunos vuelos domesticos)."
        ),
    },
    {
        "id": "ext_millas_01",
        "tema": "millas",
        "fuente_url": "https://www.latamairlines.com/es/es/centro-ayuda/preguntas/latam-pass/millas/como-acumular",
        "texto": (
            "Los pasajeros acumulan Millas LATAM Pass al volar con LATAM o con aerolineas asociadas, "
            "asi como en comercios asociados o mediante tarjetas de credito con convenio. Los pasajes "
            "que fueron pagados totalmente con millas no generan acumulacion de nuevas millas ni de "
            "Puntos Calificables."
        ),
    },
    {
        "id": "ext_millas_02",
        "tema": "millas",
        "fuente_url": "https://latampass.latam.com/es_cl/reglamento-2025/acumulacion-y-canje",
        "texto": (
            "El canje de equipaje adicional con Millas LATAM Pass solo puede realizarse dentro del "
            "mismo flujo de canje del pasaje; no es posible canjear equipaje antes o despues de haber "
            "canjeado el pasaje asociado. La cantidad de millas necesarias para un canje depende del "
            "destino y la cabina seleccionada."
        ),
    },
]

chunks_politicas = [d["texto"] for d in documentos_externos]
print(f"{len(chunks_politicas)} chunks de politicas cargados")


5 chunks de politicas cargados


## MÓDULO 3 — Motor RAG: Vectorización y Recuperación Dual

**Alcance técnico:**
- Construcción de índices vectoriales FAISS con metadata enriquecida (`Document`).
- Arquitectura de recuperación dual: búsqueda determinística/exacta para datos de reserva (privacidad) y búsqueda semántica para políticas oficiales.
- Trazabilidad de fuentes hacia URLs del Centro de Ayuda.

## 7. Bases de datos vectoriales (FAISS)

Igual que en `RA1/IL1.3/4-vector-rag.ipynb`, usamos FAISS para indexar los chunks, pero con dos
mejoras respecto a la versión inicial:

1. **Dos índices separados** (reservas y políticas): decisión de diseño para no mezclar datos
   personales de la reserva con contenido público en la misma búsqueda no controlada (justificado
   en el informe, IE3/IE8).
2. **Metadata en el índice de políticas** (`fuente_url`, `tema`, `id`): sin esto, el vector store
   solo guarda el texto plano y se pierde la URL de origen de cada chunk. Con metadata, cada
   resultado recuperado trae consigo su fuente exacta, que es justo lo que la propuesta pide en
   la sección 6 ("respuesta personalizada **y trazable a la fuente exacta**").


In [58]:
# === Validacion temprana de credenciales ===
llm_api_key = (os.getenv("LLM_API_KEY") or "").strip()
llm_base_url = (os.getenv("LLM_BASE_URL") or "https://api.groq.com/openai/v1").strip()
llm_model = (os.getenv("LLM_MODEL") or "llama-3.3-70b-versatile").strip()

if not llm_api_key:
    print("\n" + "=" * 70)
    print("ERROR: Falta la clave LLM_API_KEY (Groq).")
    print("Crea tu clave gratuita en https://console.groq.com/keys")
    print("Luego en Colab: icono llave > Nuevo Secret")
    print("  Nombre: LLM_API_KEY  |  Valor: gsk_...")
    print("  Activa el interruptor 'Notebook access'.")
    print("=" * 70 + "\n")
    raise ValueError("LLM_API_KEY no configurada.")

print(f"[OK] LLM_API_KEY detectada (inicio: {llm_api_key[:8]}...)")
print(f"[OK] LLM_BASE_URL: {llm_base_url}")
print(f"[OK] LLM_MODEL: {llm_model}")
print()

# Forzar las variables de entorno que LangChain-OpenAI lee internamente
# Esto previene que _resolve_gateway_config apunte a OpenAI en vez de Groq
os.environ["OPENAI_API_KEY"] = llm_api_key
os.environ["OPENAI_API_BASE"] = llm_base_url

# === LLM (Groq via endpoint OpenAI-compatible) ===
llm = ChatOpenAI(
    base_url=llm_base_url,
    openai_api_key=llm_api_key,
    model=llm_model,
    temperature=0.2,
    max_tokens=500,
)

# === Embeddings (local, HuggingFace) ===
# Fallback: la cuenta de Google/Gemini quedó bloqueada (403 PERMISSION_DENIED
# tras probar varias claves), así que se usa un modelo de embeddings local
# multilingue en vez de depender de la API de Gemini.
!pip install -q langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("[OK] Embeddings locales (HuggingFace) listos como fallback de Gemini.")
print("[OK] LLM (Groq) y Embeddings (HuggingFace) listos.\n")

[OK] LLM_API_KEY detectada (inicio: gsk_XHVJ...)
[OK] LLM_BASE_URL: https://api.groq.com/openai/v1
[OK] LLM_MODEL: llama-3.3-70b-versatile



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[OK] Embeddings locales (HuggingFace) listos como fallback de Gemini.
[OK] LLM (Groq) y Embeddings (HuggingFace) listos.



## 8. Retrievers y detección de código de reserva

Dos mecanismos de recuperación, cada uno para lo que le corresponde:

- **Reservas → coincidencia EXACTA.** Si la consulta trae un código de reserva (ej. `ABC123`) que
  existe en los datos, se devuelve ese perfil exacto en vez de hacer búsqueda semántica. Es más
  seguro (no hay ambigüedad de "qué reserva es la más parecida") y evita mezclar el perfil de un
  pasajero con el de otro.
- **Políticas → búsqueda SEMÁNTICA** vía `as_retriever()`, igual que en el notebook de clase, pero
  ahora cada resultado devuelve también su `fuente_url` y `tema` (gracias a la metadata agregada
  en la sección 7), no solo el texto plano.

Al final se incluye una prueba rápida de ambos mecanismos, para dejar evidencia en el informe de
que la recuperación funciona antes de conectarla al LLM (Bloque 4).


In [59]:
def detectar_codigo_reserva(query: str):
    """Busca un patrón de código de reserva (3 letras + 3 números, ej. ABC123) en la consulta."""
    match = re.search(r"\b[A-Z]{3}\d{3}\b", query.upper())
    return match.group(0) if match else None


def recuperar_contexto_reserva(query: str):
    """Si la consulta trae un código de reserva válido y existente en df_reservas, devuelve
    su perfil narrativo exacto (sin pasar por FAISS: es una búsqueda determinística, no
    semántica). Si no hay código o no existe, devuelve listas/valores vacíos para que el
    agente (Bloque 4) sepa que debe pedir el dato al pasajero en vez de asumirlo."""
    codigo = detectar_codigo_reserva(query)
    if codigo and codigo in df_reservas["codigo_reserva"].values:
        fila = df_reservas[df_reservas["codigo_reserva"] == codigo].iloc[0]
        return [fila["perfil_narrativo"]], codigo
    return [], None


politicas_retriever = politicas_db.as_retriever(search_kwargs={"k": 2})


def recuperar_contexto_politicas(query: str):
    """Recupera los chunks de políticas más relevantes para la consulta.

    Devuelve una lista de dicts {"texto", "fuente_url", "tema"} en vez de solo texto plano:
    esto es lo que permite que el agente (Bloque 4) cite la fuente exacta de cada dato en su
    respuesta, cumpliendo el requisito de trazabilidad de la propuesta (sección 6).
    """
    docs = politicas_retriever.invoke(query)
    return [
        {
            "texto": d.page_content,
            "fuente_url": d.metadata.get("fuente_url", "fuente no disponible"),
            "tema": d.metadata.get("tema", "sin tema"),
        }
        for d in docs
    ]


_codigo_ejemplo = df_reservas.iloc[0]["codigo_reserva"]

print("[Test] Prueba 1 — recuperación EXACTA por código de reserva:")
_ctx_reserva, _codigo_detectado = recuperar_contexto_reserva(f"tengo la reserva {_codigo_ejemplo}")
print(f"  código detectado: {_codigo_detectado}")
print(f"  contexto recuperado: {_ctx_reserva}")

print("\n[Test] Prueba 2 — recuperación SEMÁNTICA de políticas (con fuente citada):")
for _chunk in recuperar_contexto_politicas("¿cuánto equipaje puedo llevar?"):
    print(f"  - [{_chunk['tema']}] {_chunk['texto'][:90]}...")
    print(f"    Fuente: {_chunk['fuente_url']}")

print("\n[Test] Prueba 3 — consulta sin código de reserva (debe devolver contexto vacío):")
_ctx_vacio, _codigo_vacio = recuperar_contexto_reserva("¿puedo cambiar mi vuelo de mañana?")
assert _ctx_vacio == [] and _codigo_vacio is None, "Se esperaba contexto vacío sin código de reserva"
print("  OK: sin código de reserva -> contexto vacío (el agente deberá pedir el dato).")


NameError: name 'politicas_db' is not defined

## MÓDULO 4 — Agente Conversacional, Prompt Engineering y Evaluación
**Responsable:** Rafael Pacheco (Integrante 2)

**Alcance técnico:**
- Formulación del `SYSTEM_PROMPT` con restricciones de no alucinación y verificación de vigencia.
- Ensamble del prompt aumentado multi-fuente (reserva + políticas + consulta).
- Función orquestadora `consultar_agente()` y ejecución de casos de prueba demostrativos.


## 9. Prompt aumentado y llamada al LLM

En vez de `RetrievalQA` (pensado para una sola fuente), armamos el prompt manualmente para poder
combinar **dos fuentes** (reserva + políticas) en un mismo contexto, siguiendo los componentes de
un prompt efectivo vistos en clase (rol, tarea, formato, restricciones, input).

In [ ]:
SYSTEM_PROMPT = """Eres un asistente de atencion al pasajero para LATAM Airlines.
Tu funcion es responder consultas sobre cambios de vuelo, equipaje y Millas LATAM Pass.

Reglas:
- Usa SOLO la informacion entregada en el CONTEXTO (reserva del pasajero, si existe, y politicas
  oficiales). No inventes datos ni reglas que no esten en el contexto.
- Si la consulta depende de un dato que no tienes (ej. tarifa contratada) y no hay una reserva
  asociada en el contexto, pide ese dato al pasajero en vez de asumirlo.
- Las politicas de LATAM pueden cambiar: siempre recomienda verificar la vigencia en el sitio
  oficial (latamairlines.com) antes de tomar una decision final.
- No ejecutas transacciones reales (cambios, canjes); solo informas y orientas.
- Responde en español, en un tono claro y cercano para un pasajero."""

def construir_prompt(query, contexto_reserva, contexto_politicas):
    partes = []
    if contexto_reserva:
        partes.append("=== CONTEXTO: RESERVA DEL PASAJERO ===\n" + "\n".join(contexto_reserva))
    else:
        partes.append("=== CONTEXTO: RESERVA DEL PASAJERO ===\nNo se identifico una reserva asociada a esta consulta.")

    politicas_texto = "\n---\n".join(
        f"{p['texto']}\n(Fuente: {p['fuente_url']})" for p in contexto_politicas
    )
    partes.append("=== CONTEXTO: POLITICAS OFICIALES LATAM ===\n" + politicas_texto)
    partes.append(f"=== CONSULTA DEL PASAJERO ===\n{query}")
    return "\n\n".join(partes)

def consultar_agente(query):
    contexto_reserva, codigo = recuperar_contexto_reserva(query)
    contexto_politicas = recuperar_contexto_politicas(query)
    prompt_final = construir_prompt(query, contexto_reserva, contexto_politicas)

    respuesta = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt_final),
    ])
    return respuesta.content, contexto_reserva, contexto_politicas


## 10. Demo — consultas de ejemplo

Evidencia para el informe (IE3, IE6) y la presentación (IE7, IE9).

In [ ]:
codigo_ejemplo = df_reservas.iloc[0]["codigo_reserva"]
print("Reserva de ejemplo:", codigo_ejemplo)

pregunta_1 = f"Tengo la reserva {codigo_ejemplo}, ¿cuánto equipaje puedo llevar?"
respuesta_1, ctx_r1, ctx_p1 = consultar_agente(pregunta_1)
print("PREGUNTA:", pregunta_1)
print("\nRESPUESTA DEL AGENTE:\n", respuesta_1)


In [ ]:
pregunta_2 = "¿Cómo acumulo Millas LATAM Pass si no he volado hace tiempo?"
respuesta_2, ctx_r2, ctx_p2 = consultar_agente(pregunta_2)
print("PREGUNTA:", pregunta_2)
print("\nRESPUESTA DEL AGENTE:\n", respuesta_2)


In [ ]:
pregunta_3 = "¿Puedo cambiar mi vuelo de mañana?"
respuesta_3, ctx_r3, ctx_p3 = consultar_agente(pregunta_3)
print("PREGUNTA:", pregunta_3)
print("\nRESPUESTA DEL AGENTE (sin codigo de reserva, deberia pedir mas datos):\n", respuesta_3)


## 11. Notas para el informe técnico

- Guarda capturas de las celdas de la demo (sección 10) para el informe y la presentación
  (evidencia IE6/IE9), incluyendo el caso sin código de reserva (pregunta 3), que muestra cómo el
  agente pide más datos en vez de asumirlos.
- `reservas_latam.csv` es el dataset simulado; adjúntalo en el repositorio como evidencia.
- Este notebook usa el mismo stack que `RA1/IL1.1` (LangChain + Groq) y `RA1/IL1.3` (RAG vectorial
  con FAISS + Gemini embeddings) del curso — cítalo así en la sección de metodología/herramientas
  del informe.
- Explica en el informe por qué se separaron los índices FAISS (privacidad de la reserva) y por
  qué se armó el prompt manualmente en vez de usar `RetrievalQA` (que solo soporta una fuente).
- Las referencias APA de las 5 páginas oficiales de LATAM usadas como fuente van en la sección de
  Referencias del informe.
- Si te quedas sin cuota gratuita de Groq, revisa el README principal del repo del curso: puedes
  cambiar de proveedor (ej. Mistral) sin tocar el código, solo agregando `LLM_BASE_URL` y
  `LLM_MODEL` como Secrets adicionales.